# ForgeSight — Kaggle (2×T4)

Thin driver: logic lives in the `forgesight` repo, not in cells.

**Prereqs (do once):**
1. Attach the private dataset **`forgesight-data`** (read-only mount at `/kaggle/input/forgesight-data/`, containing `train.jsonl`/`val.jsonl`/`test.jsonl` + `images/`).
2. Add a GitHub PAT to **Add-ons → Secrets** as `GH_PAT` (repo read scope).
3. Accelerator = **GPU T4 x2**, Internet = **On**.

Then Run-All. Stop after the **Overfit-8** cell and check the loss curve.

In [ ]:
# 1. Clone the private repo via PAT (never hard-code the token)
from kaggle_secrets import UserSecretsClient
pat = UserSecretsClient().get_secret("GH_PAT")
!git clone -q https://{pat}@github.com/medhulk8/forgesight.git
%cd forgesight
!git log --oneline -1

In [ ]:
# 2. Install the pinned stack (NOT torch — use Kaggle's CUDA build) + the package
!pip -q install -r requirements-kaggle.txt
!pip -q install -e .

In [ ]:
# 3. Sanity: import + data mount + GPU
# NOTE: the repo dir is named 'forgesight' and sits on sys.path, so a bare
# `import forgesight` grabs it as an EMPTY namespace package, shadowing the real
# src-layout package. Force src/ to the front so the real package wins.
import sys
for _m in [x for x in list(sys.modules) if x == "forgesight" or x.startswith("forgesight.")]:
    del sys.modules[_m]
sys.path.insert(0, "src")
import os, torch, forgesight
from forgesight import schema, coords, collator, model
print("forgesight from:", forgesight.__file__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
DATA_ROOT = "/kaggle/input/forgesight-data"
print("data files:", [f for f in os.listdir(DATA_ROOT)])

In [ ]:
# 4. Step 9 gate: 4-bit Qwen2-VL-2B loads on one T4; LoRA params only; one forward pass
from forgesight import model as M
net = M.load_model_for_training(use_4bit=True, attn="sdpa")  # prints trainable params

In [ ]:
# 5. Step 10 gate: OVERFIT-8. Train loss must fall to ~0.
from forgesight.train_sft import load_config, build_trainer
cfg = load_config("configs/sft.yaml")
cfg["data_root"] = DATA_ROOT
trainer, processor, train_ds = build_trainer(cfg, overfit=8)
trainer.train()

In [ ]:
# 6. Loss curve for the 8 examples
import matplotlib.pyplot as plt
hist = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
xs, ys = zip(*hist)
plt.figure(figsize=(6, 4)); plt.plot(xs, ys, marker="o")
plt.xlabel("step"); plt.ylabel("train loss"); plt.title("Overfit-8"); plt.grid(True); plt.show()
print("final loss:", ys[-1])

In [ ]:
# 7. Does it reproduce the 8 target JSONs verbatim?
from forgesight.train_sft import _report_overfit
_report_overfit(trainer, processor, train_ds, cfg)